# Preprocessing YOLO


## 1. Xóa ảnh lỗi ở train và chuẩn hóa về đúng định dạng của YOLO

- Xóa train/01_PCB__1030.jpg và file nhãn của ảnh này
- giữ nguyên size ảnh là 640.
- Chuyển định dạng nhãn: Nhãn hiện tại là x1 y1 x2 y2 class_id theo pixel, với class ID từ 1 đến 6. YOLO cần class_id cx cy w h đã chuẩn hóa về 0–1, với class ID từ 0 đến 5.

In [10]:
import hashlib
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

try:
    from IPython.display import display
except ImportError:  # chạy ngoài Jupyter
    display = print

# ---- Cấu hình ----
RANDOM_SEED = 42
DATA_ROOT = Path(os.environ.get("PCB_DATA_ROOT", Path.cwd() / "Data" / "raw_data")).expanduser()
OUT_DIR = Path("outputs") / "eda"
SPLITS = ("train", "valid", "test")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
CLASS_ID_BASE = 1          # DeepPCB: class ID 1-6 (0 là background)
SMALL_AREA = 32 * 32       # box nhỏ: diện tích < 32x32 px

rng = np.random.default_rng(RANDOM_SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)


def display_path(path):
    try:
        return str(Path(path).resolve().relative_to(Path.cwd()))
    except ValueError:
        return str(path)


ROOT = next((c for c in (DATA_ROOT, DATA_ROOT / "DeepPCB") if (c / "Class.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"Không thấy Class.txt trong {DATA_ROOT} hoặc {DATA_ROOT / 'DeepPCB'}")

CLASS_NAMES = [
    n.strip(" '\"")
    for n in (ROOT / "Class.txt").read_text(encoding="utf-8").strip().split(",")
    if n.strip(" '\"")
]
assert len(CLASS_NAMES) == 6, f"Kỳ vọng 6 lớp, đọc được: {CLASS_NAMES}"
print("Data root:", display_path(ROOT))
print("Class ID", {i + CLASS_ID_BASE: n for i, n in enumerate(CLASS_NAMES)})

Data root: Data\raw_data
Class ID {1: 'open', 2: 'short', 3: 'mousebite', 4: 'spur', 5: 'copper', 6: 'pin-hole'}


In [16]:
OUTPUT_DIR = Path("D:/PCB Defect Detection/Data/processed/yolo_640")
BAD_IMAGE = ("train", "01_PCB__1030.jpg")

if OUTPUT_DIR.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT_DIR}")

for split in SPLITS:
    out_split = "val" if split == "valid" else split

    image_out = OUTPUT_DIR / "images" / out_split
    label_out = OUTPUT_DIR / "labels" / out_split
    image_out.mkdir(parents=True)
    label_out.mkdir(parents=True)

    image_dir = ROOT / split / "images"
    label_dir = ROOT / split / "labels"

    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue

        if (split, image_path.name) == BAD_IMAGE:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.is_file():
            raise FileNotFoundError(label_path)

        with Image.open(image_path) as image:
            width, height = image.size

        if (width, height) != (640, 640):
            raise ValueError(f"Unexpected image size: {image_path}")

        yolo_labels = []

        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            x1, y1, x2, y2, class_id = map(int, line.split())

            if not 1 <= class_id <= len(CLASS_NAMES):
                raise ValueError(f"Invalid class ID: {class_id}")

            if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
                raise ValueError(f"Invalid bounding box: {line}")

            cx = (x1 + x2) / (2 * width)
            cy = (y1 + y2) / (2 * height)
            w = (x2 - x1) / width
            h = (y2 - y1) / height

            yolo_labels.append(
                f"{class_id - 1} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
            )

        shutil.copy2(image_path, image_out / image_path.name)

        (label_out / f"{image_path.stem}.txt").write_text(
            "\n".join(yolo_labels) + ("\n" if yolo_labels else ""),
            encoding="utf-8",
        )

print("Preprocessing completed:", OUTPUT_DIR.resolve())

Preprocessing completed: D:\PCB Defect Detection\Data\processed\yolo_640


In [17]:
import yaml

data_config = {
    "path": str(OUTPUT_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

with open(OUTPUT_DIR / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False,
        allow_unicode=True,
    )

print((OUTPUT_DIR / "data.yaml").read_text(encoding="utf-8"))

path: D:\PCB Defect Detection\Data\processed\yolo_640
train: images/train
val: images/val
test: images/test
names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole

